In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

data_dir = Path('.').resolve().parent / 'data'

# Load transaction and identity datasets, merge on TransactionID
try:
    train_transaction = pd.read_csv(data_dir / 'test_transaction.csv')
    train_identity = pd.read_csv(data_dir / 'test_identity.csv')
    df = train_transaction.merge(train_identity, on='TransactionID', how='left')
    print(f"✓ Successfully loaded transaction and identity data")
except FileNotFoundError as e:
    print(f"✗ Error: {e}")
    df = None

df.head()

✓ Successfully loaded transaction and identity data


,TransactionID,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,card6,...,id-31,id-32,id-33,id-34,id-35,id-36,id-37,id-38,DeviceType,DeviceInfo
0,3663549,18403224,31.95,W,10409,111.0,150.0,visa,226.0,debit,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,3663550,18403263,49.00,W,4272,111.0,150.0,visa,226.0,debit,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3663551,18403310,171.00,W,4476,574.0,150.0,visa,226.0,debit,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3663552,18403310,284.95,W,10989,360.0,150.0,visa,166.0,debit,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,3663553,18403317,67.95,W,18018,452.0,150.0,mastercard,117.0,debit,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# Generate column profile: data types, missing values, cardinality
col_info = pd.DataFrame({
    'column_name': df.columns,
    'data_type': df.dtypes.values,
    'missing_%': df.isnull().mean().values * 100,
    'n_unique': df.nunique().values
})

col_info = col_info.sort_values(by='missing_%', ascending=False)

col_info.head(20)

,column_name,data_type,missing_%,n_unique
416,id-24,float64,99.064519,15
417,id-25,float64,99.005508,309
418,id-26,float64,99.003929,94
399,id-07,float64,99.001561,81
400,id-08,float64,99.001561,90
413,id-21,float64,99.001561,443
414,id-22,float64,99.000969,26
419,id-27,str,99.000969,2
415,id-23,str,99.000969,3
13,dist2,float64,92.809030,1814


In [ ]:
# Display full column information
col_info

,column_name,data_type,missing_%,n_unique
416,id-24,float64,99.064519,15
417,id-25,float64,99.005508,309
418,id-26,float64,99.003929,94
399,id-07,float64,99.001561,81
400,id-08,float64,99.001561,90
...,...,...,...,...
162,V109,float64,0.000000,9
163,V110,float64,0.000000,9
164,V111,float64,0.000000,9
165,V112,float64,0.000000,9


In [ ]:
# Export column summary for reference
col_info.to_csv(data_dir / 'column_summary.csv', index=False)

In [ ]:
# Remove ID column and convert transaction time to hours
df.drop(columns=['TransactionID'], inplace=True)
df['TransactionDT'] = df['TransactionDT'] / 3600

In [ ]:
# Handle missing values: drop sparse columns, fill remainder with type-appropriate defaults
missing_ratio = df.isnull().mean()

high_missing_cols = missing_ratio[missing_ratio > 0.9].index
df.drop(columns=high_missing_cols, inplace=True)

print("Dropped high-missing:", len(high_missing_cols))

# Separate numeric and object columns for appropriate fill values
df_numeric = df.select_dtypes(include=[np.number])
df_object = df.select_dtypes(include=['object'])

# Fill numeric with sentinel value, categorical with placeholder
df[df_numeric.columns] = df[df_numeric.columns].fillna(-999)
df[df_object.columns] = df[df_object.columns].fillna('missing')

Dropped high-missing: 10


C:\Users\zibsh\AppData\Local\Temp\ipykernel_18632\1718232559.py:11: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  df_object = df.select_dtypes(include=['object'])


In [ ]:
# Remove low-variance features
low_var_cols = [col for col in df.columns if df[col].nunique() <= 1]

df.drop(columns=low_var_cols, inplace=True)

print("Dropped low variance:", len(low_var_cols))

Dropped low variance: 1


In [ ]:
# Separate numerical and categorical columns
num_cols = df.select_dtypes(include=np.number).columns
cat_cols = df.select_dtypes(include=['object', 'string']).columns

print("Numerical:", len(num_cols))
print("Categorical:", len(cat_cols))

Numerical: 392
Categorical: 29


In [ ]:
# Engineer card-level behavioral features: transaction count and rolling statistics
df = df.sort_values(['card1', 'TransactionDT'])

df['tx_count'] = df.groupby('card1').cumcount()

df['tx_amt_mean_5'] = df.groupby('card1')['TransactionAmt'].transform(
    lambda x: x.rolling(5).mean()
)

df['tx_amt_std_5'] = df.groupby('card1')['TransactionAmt'].transform(
    lambda x: x.rolling(5).std()
)

C:\Users\zibsh\AppData\Local\Temp\ipykernel_18632\198390334.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['tx_count'] = df.groupby('card1').cumcount()
C:\Users\zibsh\AppData\Local\Temp\ipykernel_18632\198390334.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['tx_amt_mean_5'] = df.groupby('card1')['TransactionAmt'].transform(
C:\Users\zibsh\AppData\Local\Temp\ipykernel_18632\198390334.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which h

In [ ]:
# Create amount-based features: log transformation, user average, deviation
df['TransactionAmt_log'] = np.log1p(df['TransactionAmt'])

df['user_mean_amt'] = df.groupby('card1')['TransactionAmt'].transform('mean')

df['amt_deviation'] = df['TransactionAmt'] - df['user_mean_amt']

C:\Users\zibsh\AppData\Local\Temp\ipykernel_18632\3284612609.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['TransactionAmt_log'] = np.log1p(df['TransactionAmt'])
C:\Users\zibsh\AppData\Local\Temp\ipykernel_18632\3284612609.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['user_mean_amt'] = df.groupby('card1')['TransactionAmt'].transform('mean')
C:\Users\zibsh\AppData\Local\Temp\ipykernel_18632\3284612609.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert`

In [ ]:
# Extract temporal features: hour and day, compute transaction frequency per day
df['hour'] = df['TransactionDT'] % 24
df['day'] = df['TransactionDT'] // 24

df['tx_per_day'] = df.groupby(['card1', 'day'])['TransactionAmt'].transform('count')

C:\Users\zibsh\AppData\Local\Temp\ipykernel_18632\2802793215.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['hour'] = df['TransactionDT'] % 24
C:\Users\zibsh\AppData\Local\Temp\ipykernel_18632\2802793215.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['day'] = df['TransactionDT'] // 24
C:\Users\zibsh\AppData\Local\Temp\ipykernel_18632\2802793215.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining a

In [ ]:
# Feature: count unique users sharing email or device (identity theft indicators)
if 'P_emaildomain' in df.columns:
    email_users = df.groupby('P_emaildomain')['card1'].nunique()
    df['email_user_count'] = df['P_emaildomain'].map(email_users)

if 'DeviceType' in df.columns:
    device_users = df.groupby('DeviceType')['card1'].nunique()
    df['device_user_count'] = df['DeviceType'].map(device_users)

C:\Users\zibsh\AppData\Local\Temp\ipykernel_18632\3721693560.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['email_user_count'] = df['P_emaildomain'].map(email_users)
C:\Users\zibsh\AppData\Local\Temp\ipykernel_18632\3721693560.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['device_user_count'] = df['DeviceType'].map(device_users)


In [ ]:
# Feature: count distinct devices and emails per card (device/email switching)
if 'DeviceType' in df.columns:
    df['device_count'] = df.groupby('card1')['DeviceType'].transform('nunique')

if 'P_emaildomain' in df.columns:
    df['email_count'] = df.groupby('card1')['P_emaildomain'].transform('nunique')

C:\Users\zibsh\AppData\Local\Temp\ipykernel_18632\592321820.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['device_count'] = df.groupby('card1')['DeviceType'].transform('nunique')
C:\Users\zibsh\AppData\Local\Temp\ipykernel_18632\592321820.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['email_count'] = df.groupby('card1')['P_emaildomain'].transform('nunique')


In [ ]:
# Feature: frequency encoding for high-cardinality categorical columns
high_card_cols = ['card1', 'card2', 'addr1', 'P_emaildomain']

for col in high_card_cols:
    if col in df.columns:
        freq = df[col].value_counts(dropna=False)
        df[col + '_freq'] = df[col].map(freq)

C:\Users\zibsh\AppData\Local\Temp\ipykernel_18632\872208917.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[col + '_freq'] = df[col].map(freq)


In [ ]:
# Encode categorical variables using label encoding
from sklearn.preprocessing import LabelEncoder

for col in cat_cols:
    df[col] = df[col].astype(str)
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])

In [ ]:
# Final data cleaning: handle infinite values and missing values
df.replace([np.inf, -np.inf], -999, inplace=True)
df.fillna(-999, inplace=True)

print("Final Shape:", df.shape)

Final Shape: (506691, 438)


In [ ]:
# Export processed dataset
df.to_csv('../data/processed_fraud_data.csv', index=False)